# 🧠 MockMate AI — Fine-Tune Your Custom Model

This notebook trains a custom AI model on your MockMate interview data.

**Requirements:**
- Google Colab with **T4 GPU** (free tier)
- Your `training_data.jsonl` file (exported from MockMate)

**What it does:**
1. Loads your interview Q&A data
2. Fine-tunes `unsloth/Phi-3.5-mini-instruct` (3.8B params)
3. Exports as GGUF model for Ollama
4. You download and run locally on F: drive

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Upload your training_data.jsonl
from google.colab import files
print("Upload your training_data.jsonl file:")
uploaded = files.upload()
TRAIN_FILE = list(uploaded.keys())[0]
print(f"✅ Loaded: {TRAIN_FILE}")

In [ ]:
# Step 3: Load and preview data
import json

data = []
with open(TRAIN_FILE, 'r') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"📊 Total samples: {len(data)}")
print(f"\n📝 Sample entry:")
print(json.dumps(data[0], indent=2)[:500])

In [ ]:
# Step 4: Load base model with Unsloth (4-bit quantized for free Colab)
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Phi-3.5-mini-instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # auto-detect
    load_in_4bit=True,  # saves VRAM
)
print(f"✅ Loaded {MODEL_NAME}")

In [ ]:
# Step 5: Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("✅ LoRA adapters added")

In [ ]:
# Step 6: Format data into Alpaca prompt template
from datasets import Dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_sample(sample):
    return {
        'text': ALPACA_PROMPT.format(
            instruction=sample['instruction'],
            input=sample.get('input', ''),
            output=sample['output']
        )
    }

dataset = Dataset.from_list(data).map(format_sample)
print(f"✅ Formatted {len(dataset)} samples")
print(f"\n📝 Preview:\n{dataset[0]['text'][:400]}...")

In [ ]:
# Step 7: Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="./mockmate-model-output",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Loss: {trainer_stats.training_loss:.4f}")
print(f"   Time: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Step 8: Test the model
FastLanguageModel.for_inference(model)

test_prompt = ALPACA_PROMPT.format(
    instruction="Generate 3 Medium difficulty MCQ interview questions about Data Structures.",
    input="",
    output=""
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.7)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the response
response = result.split("### Response:")[-1].strip()
print("🧪 Test Output:")
print(response[:1000])

In [ ]:
# Step 9: Export as GGUF for Ollama
model.save_pretrained_gguf(
    "mockmate-ai-model",
    tokenizer,
    quantization_method="q4_k_m"  # Good balance of size vs quality
)
print("✅ Model exported as GGUF")

# Download the model file
from google.colab import files
import glob
gguf_files = glob.glob("mockmate-ai-model/*.gguf")
if gguf_files:
    print(f"📥 Downloading: {gguf_files[0]}")
    files.download(gguf_files[0])
    print("\n🎉 Done! Save this file to F:\\MockMate-AI-Training\\models\\")
    print("Then follow the Ollama setup instructions in the README.")